**生成式 AI 使用声明**：就本作业而言，使用生成式 AI 工具须遵守与合作相同的政策。与与其他合作者一样，每位学生必须独立于交互输出之外写下解决方案，且提交的作业应注明合作的性质。使用生成式 AI 工具实质性完成作业的某些部分不符合作业的精神，将违反[荣誉守则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 这将把你的 Google Drive 挂载到 Colab 虚拟机。
from google.colab import drive
drive.mount('/content/drive')

# TODO: 输入你在 Drive 中保存了解压后作业文件夹的路径，
# 例如 'cs231n/assignments/assignment2/'
FOLDERNAME = 'cs231n/assignments/assignment2/'
assert FOLDERNAME is not None, "[!] 请输入文件夹名称。"

# 现在我们已经挂载了你的 Drive，这确保
# Colab 虚拟机的 Python 解释器可以从中
# 加载 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 如果 CIFAR-10 数据集不存在，这将下载它到你的 Drive。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# 卷积网络

到目前为止，我们一直在使用深度全连接网络，使用它们来探索不同的优化策略和网络架构。全连接网络是一个很好的实验测试平台，因为它们计算效率很高，但在实践中，所有最先进的结果都使用卷积网络。

首先，你将实现卷积网络中使用的几种层类型。然后，你将使用这些层在 CIFAR-10 数据集上训练一个卷积网络。

In [ ]:
# 设置单元格。
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.cnn import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient_array, eval_numerical_gradient
from cs231n.layers import *
from cs231n.fast_layers import *
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置默认绘图大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
  """ 返回相对误差 """
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# 加载（预处理后的）CIFAR-10 数据。
data = get_CIFAR10_data()
for k, v in list(data.items()):
    print(f"{k}: {v.shape}")

# 卷积：朴素前向传播
卷积网络的核心是卷积操作。在文件 `cs231n/layers.py` 中，实现函数 `conv_forward_naive` 中的卷积层前向传播。

你目前不必过于担心效率；以你觉得最清晰的方式编写代码即可。

你可以通过运行以下代码来测试你的实现：

In [ ]:
x_shape = (2, 3, 4, 4)
w_shape = (3, 3, 4, 4)
x = np.linspace(-0.1, 0.5, num=np.prod(x_shape)).reshape(x_shape)
w = np.linspace(-0.2, 0.3, num=np.prod(w_shape)).reshape(w_shape)
b = np.linspace(-0.1, 0.2, num=3)

conv_param = {'stride': 2, 'pad': 1}
out, _ = conv_forward_naive(x, w, b, conv_param)
correct_out = np.array([[[[-0.08759809, -0.10987781],
                           [-0.18387192, -0.2109216 ]],
                          [[ 0.21027089,  0.21661097],
                           [ 0.22847626,  0.23004637]],
                          [[ 0.50813986,  0.54309974],
                           [ 0.64082444,  0.67101435]]],
                         [[[-0.98053589, -1.03143541],
                           [-1.19128892, -1.24695841]],
                          [[ 0.69108355,  0.66880383],
                           [ 0.59480972,  0.56776003]],
                          [[ 2.36270298,  2.36904306],
                           [ 2.38090835,  2.38247847]]]])

# 将你的输出与我们的比较；差异应在 e-8 左右
print('测试 conv_forward_naive')
print('差异: ', rel_error(out, correct_out))

## 题外话：通过卷积进行图像处理

作为一个既有趣又能检查你的实现并更好地理解卷积层可以执行的操作类型的方法，我们将设置一个包含两幅图像的输入，并手动设置执行常见图像处理操作（灰度转换和边缘检测）的滤波器。卷积前向传播将对每个输入图像应用这些操作。然后我们可以将结果可视化为完整性检查。

In [ ]:
from imageio import imread
from PIL import Image

kitten = imread('cs231n/notebook_images/kitten.jpg')
puppy = imread('cs231n/notebook_images/puppy.jpg')
# kitten 是宽图，puppy 已经是正方形
d = kitten.shape[1] - kitten.shape[0]
kitten_cropped = kitten[:, d//2:-d//2, :]

img_size = 200   # 如果运行太慢，请将此值调小
resized_puppy = np.array(Image.fromarray(puppy).resize((img_size, img_size)))
resized_kitten = np.array(Image.fromarray(kitten_cropped).resize((img_size, img_size)))
x = np.zeros((2, 3, img_size, img_size))
x[0, :, :, :] = resized_puppy.transpose((2, 0, 1))
x[1, :, :, :] = resized_kitten.transpose((2, 0, 1))

# 设置一个包含 2 个滤波器的卷积权重，每个 3x3
w = np.zeros((2, 3, 3, 3))

# 第一个滤波器将图像转换为灰度。
# 设置滤波器的红、绿、蓝通道。
w[0, 0, :, :] = [[0, 0, 0], [0, 0.3, 0], [0, 0, 0]]
w[0, 1, :, :] = [[0, 0, 0], [0, 0.6, 0], [0, 0, 0]]
w[0, 2, :, :] = [[0, 0, 0], [0, 0.1, 0], [0, 0, 0]]

# 第二个滤波器检测蓝色通道中的水平边缘。
w[1, 2, :, :] = [[1, 2, 1], [0, 0, 0], [-1, -2, -1]]

# 偏置向量。灰度滤波器不需要偏置，
# 但对于边缘检测滤波器，我们希望向每个输出添加 128
# 以便没有任何值为负。
b = np.array([0, 128])

# 计算将 x 中的每个输入与 w 中的每个滤波器进行卷积的结果，
# 用 b 进行偏移，并将结果存储在 out 中。
out, _ = conv_forward_naive(x, w, b, {'stride': 1, 'pad': 1})

def imshow_no_ax(img, normalize=True):
    """ 一个小辅助函数，用于将图像显示为 uint8 并移除坐标轴标签 """
    if normalize:
        img_max, img_min = np.max(img), np.min(img)
        img = 255.0 * (img - img_min) / (img_max - img_min)
    plt.imshow(img.astype('uint8'))
    plt.gca().axis('off')

# 显示原始图像和卷积操作的结果
plt.subplot(2, 3, 1)
imshow_no_ax(puppy, normalize=False)
plt.title('原始图像')
plt.subplot(2, 3, 2)
imshow_no_ax(out[0, 0])
plt.title('灰度')
plt.subplot(2, 3, 3)
imshow_no_ax(out[0, 1])
plt.title('边缘')
plt.subplot(2, 3, 4)
imshow_no_ax(kitten_cropped, normalize=False)
plt.subplot(2, 3, 5)
imshow_no_ax(out[1, 0])
plt.subplot(2, 3, 6)
imshow_no_ax(out[1, 1])
plt.show()

# 卷积：朴素反向传播
在文件 `cs231n/layers.py` 中的函数 `conv_backward_naive` 中实现卷积操作的反向传播。同样，你不需要过于担心计算效率。

完成后，运行以下代码用数值梯度检查你的反向传播。

_提示：https://deeplearning.cs.cmu.edu/F21/document/recitation/Recitation5/CNN_Backprop_Recitation_5_F21.pdf 以获得一般理解。实际的朴素实现涉及嵌套 for 循环，通过使用 dw 和 dout 的单个导数直接计算 dx_padded。_

In [ ]:
np.random.seed(231)
x = np.random.randn(4, 3, 5, 5)
w = np.random.randn(2, 3, 3, 3)
b = np.random.randn(2,)
dout = np.random.randn(4, 2, 5, 5)
conv_param = {'stride': 1, 'pad': 1}

dx_num = eval_numerical_gradient_array(lambda x: conv_forward_naive(x, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_forward_naive(x, w, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_forward_naive(x, w, b, conv_param)[0], b, dout)

out, cache = conv_forward_naive(x, w, b, conv_param)
dx, dw, db = conv_backward_naive(dout, cache)

# 你的误差应在 e-8 或更小。
print('测试 conv_backward_naive 函数')
print('dx 误差: ', rel_error(dx, dx_num))
print('dw 误差: ', rel_error(dw, dw_num))
print('db 误差: ', rel_error(db, db_num))

# 最大池化：朴素前向传播
在文件 `cs231n/layers.py` 中的函数 `max_pool_forward_naive` 中实现最大池化操作的前向传播。同样，不必过于担心计算效率。

通过运行以下代码检查你的实现：

In [ ]:
x_shape = (2, 3, 4, 4)
x = np.linspace(-0.3, 0.4, num=np.prod(x_shape)).reshape(x_shape)
pool_param = {'pool_width': 2, 'pool_height': 2, 'stride': 2}

out, _ = max_pool_forward_naive(x, pool_param)

correct_out = np.array([[[[-0.26315789, -0.24842105],
                          [-0.20421053, -0.18947368]],
                         [[-0.14526316, -0.13052632],
                          [-0.08631579, -0.07157895]],
                         [[-0.02736842, -0.01263158],
                          [ 0.03157895,  0.04631579]]],
                        [[[ 0.09052632,  0.10526316],
                          [ 0.14947368,  0.16421053]],
                         [[ 0.20842105,  0.22315789],
                          [ 0.26736842,  0.28210526]],
                         [[ 0.32631579,  0.34105263],
                          [ 0.38526316,  0.4       ]]]])

# 将你的输出与我们的比较。差异应在 e-8 数量级。
print('测试 max_pool_forward_naive 函数：')
print('差异: ', rel_error(out, correct_out))

# 最大池化：朴素反向传播
在文件 `cs231n/layers.py` 中的函数 `max_pool_backward_naive` 中实现最大池化操作的反向传播。你不需要担心计算效率。

通过运行以下代码用数值梯度检查你的实现：

In [ ]:
np.random.seed(231)
x = np.random.randn(3, 2, 8, 8)
dout = np.random.randn(3, 2, 4, 4)
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

dx_num = eval_numerical_gradient_array(lambda x: max_pool_forward_naive(x, pool_param)[0], x, dout)

out, cache = max_pool_forward_naive(x, pool_param)
dx = max_pool_backward_naive(dout, cache)

# 你的误差应在 e-12 数量级
print('测试 max_pool_backward_naive 函数：')
print('dx 误差: ', rel_error(dx, dx_num))

# 快速层

使卷积和池化层快速运行可能具有挑战性。为了省去你的痛苦，我们在文件 `cs231n/fast_layers.py` 中提供了卷积和池化层前向和反向传播的快速实现。

### 执行下面的单元格，保存 notebook，然后重启运行时
快速卷积实现依赖于 Cython 扩展；要编译它，请运行下面的单元格。接下来，保存 Colab notebook（`文件 > 保存`）并**重启运行时**（`运行时 > 重启运行时`）。然后你可以从上到下重新执行前面的单元格并跳过下面的单元格，因为你只需要为编译步骤运行它一次。

In [ ]:
# 记住在执行此单元格后重启运行时！
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/
!python setup.py build_ext --inplace
%cd /content/drive/My\ Drive/$FOLDERNAME/

快速版本的卷积和池化层的 API 与你上面实现的朴素版本完全相同：前向传播接收数据、权重和参数并产生输出和一个缓存对象；反向传播接收上游导数和缓存对象并产生关于数据和权重的梯度。

**注意：** 只有当池化区域不重叠且平铺输入时，池化的快速实现才能达到最佳性能。如果不满足这些条件，那么快速池化实现不会比朴素实现快多少。

你可以通过运行以下代码来比较这些层的朴素版本和快速版本的性能：

In [ ]:
# 相对误差应在 e-9 或更小。
from cs231n.fast_layers import conv_forward_fast, conv_backward_fast
from time import time
np.random.seed(231)
x = np.random.randn(100, 3, 31, 31)
w = np.random.randn(25, 3, 3, 3)
b = np.random.randn(25,)
dout = np.random.randn(100, 25, 16, 16)
conv_param = {'stride': 2, 'pad': 1}

t0 = time()
out_naive, cache_naive = conv_forward_naive(x, w, b, conv_param)
t1 = time()
out_fast, cache_fast = conv_forward_fast(x, w, b, conv_param)
t2 = time()

print('测试 conv_forward_fast:')
print('朴素: %fs' % (t1 - t0))
print('快速: %fs' % (t2 - t1))
print('加速比: %fx' % ((t1 - t0) / (t2 - t1)))
print('差异: ', rel_error(out_naive, out_fast))

t0 = time()
dx_naive, dw_naive, db_naive = conv_backward_naive(dout, cache_naive)
t1 = time()
dx_fast, dw_fast, db_fast = conv_backward_fast(dout, cache_fast)
t2 = time()

print('\n测试 conv_backward_fast:')
print('朴素: %fs' % (t1 - t0))
print('快速: %fs' % (t2 - t1))
print('加速比: %fx' % ((t1 - t0) / (t2 - t1)))
print('dx 差异: ', rel_error(dx_naive, dx_fast))
print('dw 差异: ', rel_error(dw_naive, dw_fast))
print('db 差异: ', rel_error(db_naive, db_fast))

In [ ]:
# 相对误差应接近 0.0。
from cs231n.fast_layers import max_pool_forward_fast, max_pool_backward_fast
np.random.seed(231)
x = np.random.randn(100, 3, 32, 32)
dout = np.random.randn(100, 3, 16, 16)
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

t0 = time()
out_naive, cache_naive = max_pool_forward_naive(x, pool_param)
t1 = time()
out_fast, cache_fast = max_pool_forward_fast(x, pool_param)
t2 = time()

print('测试 pool_forward_fast:')
print('朴素: %fs' % (t1 - t0))
print('快速: %fs' % (t2 - t1))
print('加速比: %fx' % ((t1 - t0) / (t2 - t1)))
print('差异: ', rel_error(out_naive, out_fast))

t0 = time()
dx_naive = max_pool_backward_naive(dout, cache_naive)
t1 = time()
dx_fast = max_pool_backward_fast(dout, cache_fast)
t2 = time()

print('\n测试 pool_backward_fast:')
print('朴素: %fs' % (t1 - t0))
print('快速: %fs' % (t2 - t1))
print('加速比: %fx' % ((t1 - t0) / (t2 - t1)))
print('dx 差异: ', rel_error(dx_naive, dx_fast))

# 卷积"三明治"层
在之前的作业中，我们引入了"三明治"层的概念，将多个操作组合成常用的模式。在文件 `cs231n/layer_utils.py` 中，你将找到实现卷积网络中几种常用模式的三明治层。运行下面的单元格来完整性检查它们的使用。

In [ ]:
from cs231n.layer_utils import conv_relu_pool_forward, conv_relu_pool_backward
np.random.seed(231)
x = np.random.randn(2, 3, 16, 16)
w = np.random.randn(3, 3, 3, 3)
b = np.random.randn(3,)
dout = np.random.randn(2, 3, 8, 8)
conv_param = {'stride': 1, 'pad': 1}
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

out, cache = conv_relu_pool_forward(x, w, b, conv_param, pool_param)
dx, dw, db = conv_relu_pool_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], b, dout)

# 相对误差应在 e-8 或更小
print('测试 conv_relu_pool')
print('dx 误差: ', rel_error(dx_num, dx))
print('dw 误差: ', rel_error(dw_num, dw))
print('db 误差: ', rel_error(db_num, db))

In [ ]:
from cs231n.layer_utils import conv_relu_forward, conv_relu_backward
np.random.seed(231)
x = np.random.randn(2, 3, 8, 8)
w = np.random.randn(3, 3, 3, 3)
b = np.random.randn(3,)
dout = np.random.randn(2, 3, 8, 8)
conv_param = {'stride': 1, 'pad': 1}

out, cache = conv_relu_forward(x, w, b, conv_param)
dx, dw, db = conv_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: conv_relu_forward(x, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_relu_forward(x, w, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_relu_forward(x, w, b, conv_param)[0], b, dout)

# 相对误差应在 e-8 或更小
print('测试 conv_relu:')
print('dx 误差: ', rel_error(dx_num, dx))
print('dw 误差: ', rel_error(dw_num, dw))
print('db 误差: ', rel_error(db_num, db))

# 三层卷积网络
现在你已经实现了所有必要的层，我们可以将它们组合成一个简单的卷积网络。

打开文件 `cs231n/classifiers/cnn.py` 并完成 `ThreeLayerConvNet` 类的实现。记住你可以在你的实现中使用快速/三明治层（已经为你导入）。运行以下单元格来帮助你调试：

## 完整性检查损失
构建新网络后，你应该做的第一件事是完整性检查损失。当我们使用 softmax 损失时，我们期望随机权重（且无正则化）的损失约为 `log(C)`，其中 `C` 是类别数。当我们添加正则化时，损失应该略微增加。

In [ ]:
model = ThreeLayerConvNet()

N = 50
X = np.random.randn(N, 3, 32, 32)
y = np.random.randint(10, size=N)

loss, grads = model.loss(X, y)
print('初始损失（无正则化）: ', loss)

model.reg = 0.5
loss, grads = model.loss(X, y)
print('初始损失（有正则化）: ', loss)

## 梯度检查
在损失看起来合理后，使用数值梯度检查来确保你的反向传播是正确的。当你使用数值梯度检查时，应该使用少量的人工数据和每层少量的神经元。注意：正确的实现仍可能具有高达 e-2 数量级的相对误差。

In [ ]:
num_inputs = 2
input_dim = (3, 16, 16)
reg = 0.0
num_classes = 10
np.random.seed(231)
X = np.random.randn(num_inputs, *input_dim)
y = np.random.randint(num_classes, size=num_inputs)

model = ThreeLayerConvNet(
    num_filters=3,
    filter_size=3,
    input_dim=input_dim,
    hidden_dim=7,
    dtype=np.float64
)
loss, grads = model.loss(X, y)
# 误差应该很小，但正确的实现可能具有
# 高达 e-2 数量级的相对误差
for param_name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    param_grad_num = eval_numerical_gradient(f, model.params[param_name], verbose=False, h=1e-6)
    e = rel_error(param_grad_num, grads[param_name])
    print('%s 最大相对误差: %e' % (param_name, rel_error(param_grad_num, grads[param_name])))

## 在小数据上过拟合
一个很好的技巧是只用少量训练样本训练你的模型。你应该能够过拟合小数据集，这将导致非常高的训练准确率和相对较低的验证准确率。

In [ ]:
np.random.seed(231)

num_train = 100
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

model = ThreeLayerConvNet(weight_scale=1e-2)

solver = Solver(
    model,
    small_data,
    num_epochs=15,
    batch_size=50,
    update_rule='adam',
    optim_config={'learning_rate': 1e-3,},
    verbose=True,
    print_every=1
)
solver.train()

In [ ]:
# 打印最终训练准确率。
print(
    "小数据训练准确率:",
    solver.check_accuracy(small_data['X_train'], small_data['y_train'])
)

In [ ]:
# 打印最终验证准确率。
print(
    "小数据验证准确率:",
    solver.check_accuracy(small_data['X_val'], small_data['y_val'])
)

绘制损失、训练准确率和验证准确率应显示明显的过拟合：

In [ ]:
plt.subplot(2, 1, 1)
plt.plot(solver.loss_history, 'o')
plt.xlabel('迭代')
plt.ylabel('损失')

plt.subplot(2, 1, 2)
plt.plot(solver.train_acc_history, '-o')
plt.plot(solver.val_acc_history, '-o')
plt.legend(['训练', '验证'], loc='upper left')
plt.xlabel('轮次')
plt.ylabel('准确率')
plt.show()

## 训练网络
通过训练三层卷积网络一个轮次，你应该在训练集上达到超过 40% 的准确率：

In [ ]:
model = ThreeLayerConvNet(weight_scale=0.001, hidden_dim=500, reg=0.001)

solver = Solver(
    model,
    data,
    num_epochs=1,
    batch_size=50,
    update_rule='adam',
    optim_config={'learning_rate': 1e-3,},
    verbose=True,
    print_every=20
)
solver.train()

In [ ]:
# 打印最终训练准确率。
print(
    "完整数据训练准确率:",
    solver.check_accuracy(data['X_train'], data['y_train'])
)

In [ ]:
# 打印最终验证准确率。
print(
    "完整数据验证准确率:",
    solver.check_accuracy(data['X_val'], data['y_val'])
)

## 可视化滤波器
你可以通过运行以下代码来可视化训练网络的第一层卷积滤波器：

In [ ]:
from cs231n.vis_utils import visualize_grid

grid = visualize_grid(model.params['W1'].transpose(0, 2, 3, 1))
plt.imshow(grid.astype('uint8'))
plt.axis('off')
plt.gcf().set_size_inches(5, 5)
plt.show()

# 空间批归一化（Spatial Batch Normalization）
我们已经看到批归一化对于训练深度全连接网络非常有用。正如原始论文中提出的（链接在 `BatchNormalization.ipynb` 中），批归一化也可以用于卷积网络，但我们需要稍微调整一下；这种修改将被称为"空间批归一化"。

通常，批归一化接受形状为 `(N, D)` 的输入并产生形状为 `(N, D)` 的输出，其中我们在小批次维度 `N` 上进行归一化。对于来自卷积层的数据，批归一化需要接受形状为 `(N, C, H, W)` 的输入并产生形状为 `(N, C, H, W)` 的输出，其中 `N` 维度给出小批次大小，`(H, W)` 维度给出特征图的空间大小。

如果特征图是使用卷积产生的，那么我们期望每个特征通道的统计信息（例如均值、方差）在不同的图像之间以及同一图像内的不同位置之间相对一致——毕竟，每个特征通道都是由同一个卷积滤波器产生的！因此，空间批归一化通过对小批次维度 `N` 以及空间维度 `H` 和 `W` 计算统计信息来计算每个 `C` 特征通道的均值和方差。


[1] [Sergey Ioffe and Christian Szegedy, "Batch Normalization: Accelerating Deep Network Training by Reducing
Internal Covariate Shift", ICML 2015.](https://arxiv.org/abs/1502.03167)

# 空间批归一化：前向传播

在文件 `cs231n/layers.py` 中，实现函数 `spatial_batchnorm_forward` 中的空间批归一化前向传播。通过运行以下代码检查你的实现：

In [ ]:
np.random.seed(231)

# 通过检查空间批归一化前后特征的均值和方差来检查训练时前向传播。
N, C, H, W = 2, 3, 4, 5
x = 4 * np.random.randn(N, C, H, W) + 10

print('空间批归一化前：')
print('  形状: ', x.shape)
print('  均值: ', x.mean(axis=(0, 2, 3)))
print('  标准差: ', x.std(axis=(0, 2, 3)))

# 均值应接近零，标准差接近一
gamma, beta = np.ones(C), np.zeros(C)
bn_param = {'mode': 'train'}
out, _ = spatial_batchnorm_forward(x, gamma, beta, bn_param)
print('空间批归一化后：')
print('  形状: ', out.shape)
print('  均值: ', out.mean(axis=(0, 2, 3)))
print('  标准差: ', out.std(axis=(0, 2, 3)))

# 均值应接近 beta，标准差接近 gamma
gamma, beta = np.asarray([3, 4, 5]), np.asarray([6, 7, 8])
out, _ = spatial_batchnorm_forward(x, gamma, beta, bn_param)
print('空间批归一化后（非平凡的 gamma, beta）：')
print('  形状: ', out.shape)
print('  均值: ', out.mean(axis=(0, 2, 3)))
print('  标准差: ', out.std(axis=(0, 2, 3)))

In [ ]:
np.random.seed(231)

# 通过多次运行训练时前向传播来预热运行平均值，然后
# 检查测试时前向传播后激活的均值和方差，以此来检查测试时前向传播。
N, C, H, W = 10, 4, 11, 12

bn_param = {'mode': 'train'}
gamma = np.ones(C)
beta = np.zeros(C)
for t in range(50):
  x = 2.3 * np.random.randn(N, C, H, W) + 13
  spatial_batchnorm_forward(x, gamma, beta, bn_param)
bn_param['mode'] = 'test'
x = 2.3 * np.random.randn(N, C, H, W) + 13
a_norm, _ = spatial_batchnorm_forward(x, gamma, beta, bn_param)

# 均值应接近零，标准差接近一，但会比训练时前向传播更嘈杂。
print('空间批归一化后（测试时）：')
print('  均值: ', a_norm.mean(axis=(0, 2, 3)))
print('  标准差: ', a_norm.std(axis=(0, 2, 3)))

# 空间批归一化：反向传播
在文件 `cs231n/layers.py` 中，实现函数 `spatial_batchnorm_backward` 中的空间批归一化反向传播。运行以下代码使用数值梯度检查你的实现：

In [ ]:
np.random.seed(231)
N, C, H, W = 2, 3, 4, 5
x = 5 * np.random.randn(N, C, H, W) + 12
gamma = np.random.randn(C)
beta = np.random.randn(C)
dout = np.random.randn(N, C, H, W)

bn_param = {'mode': 'train'}
fx = lambda x: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]
fg = lambda a: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]
fb = lambda b: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma, dout)
db_num = eval_numerical_gradient_array(fb, beta, dout)

# 你应该期望误差量级在 1e-12~1e-06 之间
_, cache = spatial_batchnorm_forward(x, gamma, beta, bn_param)
dx, dgamma, dbeta = spatial_batchnorm_backward(dout, cache)
print('dx 误差: ', rel_error(dx_num, dx))
print('dgamma 误差: ', rel_error(da_num, dgamma))
print('dbeta 误差: ', rel_error(db_num, dbeta))

# 空间组归一化（Spatial Group Normalization）
在之前的 notebook 中，我们提到层归一化是一种缓解批归一化批次大小限制的另一归一化技术。然而，正如 [2] 的作者所观察到的，当与卷积层一起使用时，层归一化的表现不如批归一化：

>对于全连接层，一层中的所有隐藏单元倾向于对最终预测做出相似的贡献，对一层的输入之和进行重新定中心和重新缩放效果很好。然而，相似的贡献假设对于卷积神经网络不再成立。大量感受野位于图像边界附近的隐藏单元很少被激活，因此具有与同一层中其余隐藏单元非常不同的统计信息。

[3] 的作者提出了一种中间技术。与层归一化不同，在层归一化中，你对每个数据点的整个特征进行归一化，他们建议将每个数据点的特征一致地分成 G 组，并进行每组每个数据点的归一化。

<p align="center">
<img src="https://raw.githubusercontent.com/cs231n/cs231n.github.io/master/assets/a2/normalization.png">
</p>
<center>到目前为止讨论的归一化技术的视觉比较（图片来自 [3]）</center>

即使在每组内仍然做出了相等的贡献假设，作者假设这并不会有太大问题，因为特征中会出现用于视觉识别的天生分组。他们用来说明这一点的一个例子是，传统计算机视觉中许多高性能的手工特征都有明确分组在一起的项。例如，定向梯度直方图 [4]——在计算每个空间局部块的直方图后，每个块直方图在被连接在一起形成最终特征向量之前会被归一化。

你现在将实现组归一化。

[2] [Ba, Jimmy Lei, Jamie Ryan Kiros, and Geoffrey E. Hinton. "Layer Normalization." stat 1050 (2016): 21.](https://arxiv.org/pdf/1607.06450.pdf)


[3] [Wu, Yuxin, and Kaiming He. "Group Normalization." arXiv preprint arXiv:1803.08494 (2018).](https://arxiv.org/abs/1803.08494)


[4] [N. Dalal and B. Triggs. Histograms of oriented gradients for
human detection. In Computer Vision and Pattern Recognition
(CVPR), 2005.](https://ieeexplore.ieee.org/abstract/document/1467360/)

# 空间组归一化：前向传播

在文件 `cs231n/layers.py` 中，实现函数 `spatial_groupnorm_forward` 中的组归一化前向传播。通过运行以下代码检查你的实现：

In [ ]:
np.random.seed(231)

# 通过检查空间组归一化前后特征的均值和方差来检查训练时前向传播。
N, C, H, W = 2, 6, 4, 5
G = 2
x = 4 * np.random.randn(N, C, H, W) + 10
x_g = x.reshape((N*G,-1))
print('空间组归一化前：')
print('  形状: ', x.shape)
print('  均值: ', x_g.mean(axis=1))
print('  标准差: ', x_g.std(axis=1))

# 均值应接近零，标准差接近一
gamma, beta = np.ones((1,C,1,1)), np.zeros((1,C,1,1))
bn_param = {'mode': 'train'}

out, _ = spatial_groupnorm_forward(x, gamma, beta, G, bn_param)
out_g = out.reshape((N*G,-1))
print('空间组归一化后：')
print('  形状: ', out.shape)
print('  均值: ', out_g.mean(axis=1))
print('  标准差: ', out_g.std(axis=1))

# 空间组归一化：反向传播
在文件 `cs231n/layers.py` 中，实现函数 `spatial_groupnorm_backward` 中的组归一化反向传播。运行以下代码使用数值梯度检查你的实现：

In [ ]:
np.random.seed(231)
N, C, H, W = 2, 6, 4, 5
G = 2
x = 5 * np.random.randn(N, C, H, W) + 12
gamma = np.random.randn(1,C,1,1)
beta = np.random.randn(1,C,1,1)
dout = np.random.randn(N, C, H, W)

gn_param = {}
fx = lambda x: spatial_groupnorm_forward(x, gamma, beta, G, gn_param)[0]
fg = lambda a: spatial_groupnorm_forward(x, gamma, beta, G, gn_param)[0]
fb = lambda b: spatial_groupnorm_forward(x, gamma, beta, G, gn_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma, dout)
db_num = eval_numerical_gradient_array(fb, beta, dout)

_, cache = spatial_groupnorm_forward(x, gamma, beta, G, gn_param)
dx, dgamma, dbeta = spatial_groupnorm_backward(dout, cache)

# 你应该期望误差量级在 1e-12 到 1e-07 之间。
print('dx 误差: ', rel_error(dx_num, dx))
print('dgamma 误差: ', rel_error(da_num, dgamma))
print('dbeta 误差: ', rel_error(db_num, dbeta))